In [4]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time

# Setup
options = Options()
options.add_argument("--headless=new")
options.add_argument("--window-size=1920,1080")
driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# URL
program = "Bachelor i ingeniørfag, dataingeniør"
url = "https://www.usn.no/studier/studie-og-emneplaner/#/studieplan/ING2_2025_H%C3%98ST"

print(f"\n🔍 Processing: {program}")
driver.get("https://www.usn.no/studier/studie-og-emneplaner/")
time.sleep(3)

# Accept cookies
try:
    cookie_button = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Godta alle')]"))
    )
    print("🍪 Clicking 'Godta alle'...")
    cookie_button.click()
    time.sleep(2)
except Exception as e:
    print("⚠️ No cookie popup or couldn't click:", e)

# Navigate
hash_part = url.split("#")[-1]
driver.execute_script(f"window.location.hash = '{hash_part}'")
print("⏳ Waiting 10s for full page load...")
time.sleep(10)

# Scroll to bottom
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
time.sleep(2)

# Save snapshot
driver.save_screenshot("valgemner_test.png")
html = driver.execute_script("return document.documentElement.outerHTML;")
with open("valgemner_test.html", "w", encoding="utf-8") as f:
    f.write(html)

# Parse HTML
soup = BeautifulSoup(html, "html.parser")

# Find the "Valgemner" header and extract content until "Studiemodell"
valgemner = soup.find(string=lambda t: t and "valgemner" in t.lower())
studiemodell = soup.find(string=lambda t: t and "studiemodell" in t.lower())

if valgemner and studiemodell:
    content = []
    current = valgemner.find_parent()
    while current and studiemodell not in current.get_text():
        content.append(current.get_text(strip=True))
        current = current.find_next_sibling()
    
    print("\n🧾 Valgemner content found:")
    for line in content:
        print("-", line)
else:
    print("❌ Could not locate both 'Valgemner' and 'Studiemodell'")

driver.quit()



🔍 Processing: Bachelor i ingeniørfag, dataingeniør
🍪 Clicking 'Godta alle'...
⏳ Waiting 10s for full page load...
❌ Could not locate both 'Valgemner' and 'Studiemodell'
